# Análisis del portafolio — ABC Analytics

**Proyecto:** Migración cloud de la API de predicción de ventas
**Autores:** Juan Campos y Mónica Penacho
**Asignatura:** 20GIAR · VIU
**Fecha:** Mayo 2026

---

Dado el portafolio on-premise de la **empresa ficticia ABC Analytics**, como arquitectos cloud clasificamos cada aplicación con las 6 R's de migración, seleccionamos la más interesante para diseñar su arquitectura cloud objetivo, y construimos el plan de waves de migración:

1. **Portafolio de aplicaciones** on-premise
2. **Motor de clasificación con las 6 R's**
3. **Análisis del portafolio** — qué se migra, qué no y con qué estrategia
4. **Diseño de la arquitectura cloud objetivo** — para la app seleccionada (Sales Prediction API)
5. **Wave Planning** — orden y dependencias de la migración
6. **Conclusión y aplicación al proyecto**

El resultado justifica que **la `Sales Prediction API` se migre con estrategia `Replatform`**, que es exactamente la app que se implementa y se despliega en Azure en el resto del repositorio.

---
## 1. Portafolio de aplicaciones

Empresa ficticia: **ABC Analytics, S.L.** — empresa de retail analytics con 6 años de actividad, ~120 empleados, en proceso de migración al cloud (Azure).

El portafolio on-premise está formado por **5 aplicaciones**. Para cada una se recogen los criterios necesarios para decidir la estrategia de migración.

In [1]:
aplicaciones = [
    {
        "nombre": "Sales Prediction API",
        "descripcion": "API REST que predice ventas mensuales por producto",
        "tecnologia": "Python 3.11, FastAPI",
        "criticidad": "alta",
        "uso_mensual_usuarios": 12000,
        "datos_sensibles": False,
        "complejidad_tecnica": "media",
        "equipo_conocimiento_cloud": "medio"
    },
    {
        "nombre": "ETL nocturno de ventas",
        "descripcion": "Job Python que carga ventas a un Postgres on-premise",
        "tecnologia": "Python 3.8, PostgreSQL 12",
        "criticidad": "alta",
        "uso_mensual_usuarios": 0,
        "datos_sensibles": True,
        "complejidad_tecnica": "media",
        "equipo_conocimiento_cloud": "bajo"
    },
    {
        "nombre": "Dashboard KPIs Retail",
        "descripcion": "Panel interno de KPIs comerciales para dirección",
        "tecnologia": "Tableau Server (on-premise)",
        "criticidad": "baja",
        "uso_mensual_usuarios": 35,
        "datos_sensibles": False,
        "complejidad_tecnica": "baja",
        "equipo_conocimiento_cloud": "alto"
    },
    {
        "nombre": "Reportes legacy clientes",
        "descripcion": "Sistema antiguo de reporting para clientes B2B",
        "tecnologia": "Crystal Reports, IIS, Windows Server 2008",
        "criticidad": "baja",
        "uso_mensual_usuarios": 4,
        "datos_sensibles": False,
        "complejidad_tecnica": "alta",
        "equipo_conocimiento_cloud": "bajo"
    },
    {
        "nombre": "Modelo scoring de clientes",
        "descripcion": "Modelo legacy en R que segmenta clientes por riesgo",
        "tecnologia": "R 3.6, SQL Server 2012 (EOL)",
        "criticidad": "muy alta",
        "uso_mensual_usuarios": 0,
        "datos_sensibles": True,
        "complejidad_tecnica": "media",
        "equipo_conocimiento_cloud": "bajo"
    },
]

print(f"Portafolio cargado: {len(aplicaciones)} aplicaciones\n")
for app in aplicaciones:
    print(f"  • {app['nombre']}")
    print(f"    Tecnología : {app['tecnologia']}")
    print(f"    Criticidad : {app['criticidad']:10} | Usuarios/mes: {app['uso_mensual_usuarios']}")
    print(f"    Datos sens.: {'Sí' if app['datos_sensibles'] else 'No':10} | Complejidad: {app['complejidad_tecnica']}\n")

Portafolio cargado: 5 aplicaciones

  • Sales Prediction API
    Tecnología : Python 3.11, FastAPI
    Criticidad : alta       | Usuarios/mes: 12000
    Datos sens.: No         | Complejidad: media

  • ETL nocturno de ventas
    Tecnología : Python 3.8, PostgreSQL 12
    Criticidad : alta       | Usuarios/mes: 0
    Datos sens.: Sí         | Complejidad: media

  • Dashboard KPIs Retail
    Tecnología : Tableau Server (on-premise)
    Criticidad : baja       | Usuarios/mes: 35
    Datos sens.: No         | Complejidad: baja

  • Reportes legacy clientes
    Tecnología : Crystal Reports, IIS, Windows Server 2008
    Criticidad : baja       | Usuarios/mes: 4
    Datos sens.: No         | Complejidad: alta

  • Modelo scoring de clientes
    Tecnología : R 3.6, SQL Server 2012 (EOL)
    Criticidad : muy alta   | Usuarios/mes: 0
    Datos sens.: Sí         | Complejidad: media



---
## 2. Motor de clasificación — las 6 R's

Las **6 R's de migración** (Gartner / AWS) son las estrategias posibles:

| Estrategia | Descripción | Esfuerzo | Beneficio cloud |
|------------|-------------|----------|-----------------|
| **Retire** | Dar de baja la aplicación | — | — |
| **Retain** | Mantener on-premise (por ahora) | — | — |
| **Rehost** | Lift & Shift — mover sin cambios | Bajo | Bajo |
| **Replatform** | Migrar con pequeñas mejoras | Medio | Medio |
| **Repurchase** | Cambiar por SaaS equivalente | Variable | Alto |
| **Refactor** | Rediseñar nativo cloud | Alto | Muy alto |

La función siguiente implementa Motor de clasificación con las 6 R's.

In [2]:
def evaluar_migracion(app):
    """Clasifica una aplicación con la estrategia de migración más adecuada.
    Reglas idénticas a las del notebook docente 20GIAR_MIGRACION_CLOUD.ipynb."""

    # Retire: alta complejidad + bajo uso real = no vale la pena migrar
    if app["complejidad_tecnica"] == "alta" and app["uso_mensual_usuarios"] < 10:
        return {
            "estrategia": "Retire",
            "razon": "Alta complejidad técnica + uso mínimo → candidato a dar de baja",
            "siguiente_paso": "Verificar que los pocos usuarios tengan alternativa antes de apagar"
        }

    # Repurchase: existe SaaS equivalente
    if "Tableau" in app["tecnologia"] or "Crystal Reports" in app["tecnologia"]:
        return {
            "estrategia": "Repurchase",
            "razon": "Existe SaaS equivalente — elimina toda gestión de infraestructura",
            "siguiente_paso": "Evaluar Power BI / Tableau Cloud; planificar migración de dashboards"
        }

    # Retain: criticidad muy alta + datos sensibles + equipo sin experiencia cloud
    if (app["criticidad"] == "muy alta"
            and app["datos_sensibles"]
            and app["equipo_conocimiento_cloud"] == "bajo"):
        return {
            "estrategia": "Retain (por ahora)",
            "razon": "Riesgo demasiado alto con el nivel actual de experiencia cloud del equipo",
            "siguiente_paso": "Migrar en Wave 3 tras ganar experiencia con apps menos críticas"
        }

    # Replatform: complejidad media + equipo con algo de conocimiento
    if (app["complejidad_tecnica"] == "media"
            and app["equipo_conocimiento_cloud"] in ["medio", "alto"]):
        return {
            "estrategia": "Replatform",
            "razon": "Equilibrio óptimo: pequeños cambios, grandes beneficios operacionales",
            "siguiente_paso": "Contenerizar (Docker), publicar en ACR y desplegar en Azure Container Instance"
        }

    # Rehost: caso por defecto
    return {
        "estrategia": "Rehost",
        "razon": "Migración rápida sin modificar el software; optimizar en waves posteriores",
        "siguiente_paso": "Migrar a una Azure VM tal cual y revisar en 6 meses"
    }


print("Motor de clasificación cargado ✓")

Motor de clasificación cargado ✓


---
## 3. Análisis del portafolio

Aplicamos el motor a las 5 aplicaciones de ABC Analytics y obtenemos la estrategia recomendada para cada una.

In [3]:
print("ANÁLISIS DE PORTAFOLIO — ABC Analytics, S.L.")
print("=" * 68)

resultados = []
for app in aplicaciones:
    resultado = evaluar_migracion(app)
    resultados.append({**app, **resultado})

    print(f"\n{'─'*68}")
    print(f"APLICACIÓN : {app['nombre']}")
    print(f"Tecnología : {app['tecnologia']}")
    print(f"Criticidad : {app['criticidad']:10} | Usuarios/mes: {app['uso_mensual_usuarios']}")
    print(f"Datos sens.: {'Sí' if app['datos_sensibles'] else 'No':10} | Complejidad: {app['complejidad_tecnica']}")
    print(f"\n  ➜  Estrategia   : [{resultado['estrategia']}]")
    print(f"     Razón         : {resultado['razon']}")
    print(f"     Siguiente paso: {resultado['siguiente_paso']}")

# Resumen
from collections import Counter
conteo = Counter(r['estrategia'] for r in resultados)
print(f"\n{'='*68}")
print("RESUMEN DEL PORTAFOLIO:")
for estrategia, n in conteo.items():
    print(f"  {estrategia:25}: {n} app{'s' if n > 1 else ''}")

ANÁLISIS DE PORTAFOLIO — ABC Analytics, S.L.

────────────────────────────────────────────────────────────────────
APLICACIÓN : Sales Prediction API
Tecnología : Python 3.11, FastAPI
Criticidad : alta       | Usuarios/mes: 12000
Datos sens.: No         | Complejidad: media

  ➜  Estrategia   : [Replatform]
     Razón         : Equilibrio óptimo: pequeños cambios, grandes beneficios operacionales
     Siguiente paso: Contenerizar (Docker), publicar en ACR y desplegar en Azure Container Instance

────────────────────────────────────────────────────────────────────
APLICACIÓN : ETL nocturno de ventas
Tecnología : Python 3.8, PostgreSQL 12
Criticidad : alta       | Usuarios/mes: 0
Datos sens.: Sí         | Complejidad: media

  ➜  Estrategia   : [Rehost]
     Razón         : Migración rápida sin modificar el software; optimizar en waves posteriores
     Siguiente paso: Migrar a una Azure VM tal cual y revisar en 6 meses

────────────────────────────────────────────────────────────────────


---
## 4. Diseño de la arquitectura cloud objetivo

De las 5 aplicaciones del portafolio, **sólo la Sales Prediction API se materializa en código** en este proyecto, porque es la única clasificada como `Replatform`. Por tanto, el diseño de arquitectura cloud objetivo se centra en ella.

### 4.1 Requisitos derivados de la app
La Sales Prediction API debe cumplir lo siguiente en cloud:

1. **Stateless** — la API no guarda estado, así que se puede escalar horizontalmente.
2. **HTTP público** — necesita endpoint accesible desde fuera para `POST /predict`.
3. **Versionada por commit** — trazabilidad total de qué imagen está corriendo en cada momento.
4. **Aprovisionable por IaC** — toda la infraestructura debe poder reconstruirse con `terraform apply`.
5. **Bajo coste** — restricción de cuenta Azure for Students.
6. **Aprobación humana en producción** — requisito explícito de la clase 24/04.

### 4.2 Servicios Azure elegidos

| Capa | Servicio Azure | Por qué |
|------|----------------|---------|
| **Cómputo** | Azure Container Instance (ACI) | Más simple que AKS; pago por segundo; no requiere clúster Kubernetes |
| **Registro de imagen** | Azure Container Registry (ACR Basic) | Privado, integrado con ACI; SKU Basic suficiente para una app |
| **IaC** | Terraform (provider azurerm) | Reproducible, versionable, declarativo |
| **CI/CD** | GitHub Actions | Ya integrado con el repo; gratis para repos públicos |
| **Identidad** | Service Principal de Azure AD | Autenticación no-humana segura para CI/CD |
| **Agrupación lógica** | Resource Group | Permite borrarlo todo de un comando (`az group delete`) |

### 4.3 Diagrama de arquitectura (Mermaid)

```mermaid
flowchart LR
  Dev[Desarrollador] -->|git push| GH[GitHub Repo]
  GH --> CI[GitHub Actions: CI<br/>lint + tests + build]
  CI --> CD[GitHub Actions: CD<br/>build & push imagen]
  CD --> ACR[(Azure Container Registry)]
  CD --> TF[Terraform apply]
  TF --> RG[Azure Resource Group]
  RG --> ACI[Azure Container Instance]
  ACR -->|pull imagen| ACI
  Cliente -->|HTTP POST /predict| ACI
```

### 4.4 Flujo de datos en producción

1. El cliente envía `POST /predict` con `{product, base_sales, month}` al FQDN público de ACI.
2. ACI recibe la petición en el puerto 8000 (uvicorn).
3. FastAPI valida con Pydantic (`SalesPrediction`) — devuelve `422` si la entrada es inválida.
4. La función `predict_sales` calcula el resultado (modelo simulado en este proyecto académico).
5. ACI devuelve `PredictionResponse` con `predicted_sales` y `confidence`.

> Nota MLOps: en un proyecto real, el modelo iría en Azure ML Foundry o como artefacto en ACR; aquí se simula intencionadamente para mantener el alcance académico.

In [4]:
# Captura formal de la arquitectura objetivo (auto-documentada)
arquitectura = {
    "aplicacion": "Sales Prediction API",
    "estrategia_6R": "Replatform",
    "componentes": [
        {"capa": "Resource Group",     "servicio": "azurerm_resource_group",  "sku": "-",     "rol": "Agrupacion logica de todos los recursos"},
        {"capa": "Container Registry", "servicio": "azurerm_container_registry","sku": "Basic","rol": "Almacena la imagen Docker privada de la API"},
        {"capa": "Container Instance", "servicio": "azurerm_container_group", "sku": "0.5 CPU / 1.5 GB", "rol": "Ejecuta el contenedor con IP publica"},
        {"capa": "CI/CD",              "servicio": "GitHub Actions",          "sku": "free", "rol": "Lint, tests, build y deploy automatizados"},
        {"capa": "Identidad",          "servicio": "Service Principal",       "sku": "-",    "rol": "Autenticacion de GitHub Actions contra Azure"}
    ],
    "principios_aplicados": [
        "12-Factor App: stateless + config via env vars (APP_ENV)",
        "Inmutabilidad: cada commit produce una imagen nueva (tag = SHA)",
        "Aprobacion humana en produccion (GitHub Environments)",
        "Tres vias DevOps: flujo (CI rapido), feedback (tests + lint), aprendizaje (retros)"
    ]
}

print(f"ARQUITECTURA CLOUD OBJETIVO - {arquitectura['aplicacion']}")
print(f"Estrategia 6R: {arquitectura['estrategia_6R']}")
print("=" * 68)
print(f"\n{'CAPA':25} {'SERVICIO':35} {'SKU':18}")
print("-" * 68)
for c in arquitectura["componentes"]:
    print(f"{c['capa']:25} {c['servicio']:35} {c['sku']:18}")
print("\nPRINCIPIOS APLICADOS:")
for p in arquitectura["principios_aplicados"]:
    print(f"  - {p}")

ARQUITECTURA CLOUD OBJETIVO - Sales Prediction API
Estrategia 6R: Replatform

CAPA                      SERVICIO                            SKU               
--------------------------------------------------------------------
Resource Group            azurerm_resource_group              -                 
Container Registry        azurerm_container_registry          Basic             
Container Instance        azurerm_container_group             0.5 CPU / 1.5 GB  
CI/CD                     GitHub Actions                      free              
Identidad                 Service Principal                   -                 

PRINCIPIOS APLICADOS:
  - 12-Factor App: stateless + config via env vars (APP_ENV)
  - Inmutabilidad: cada commit produce una imagen nueva (tag = SHA)
  - Aprobacion humana en produccion (GitHub Environments)
  - Tres vias DevOps: flujo (CI rapido), feedback (tests + lint), aprendizaje (retros)


### 4.5 Decisiones de diseño justificadas

| Decisión | Alternativa descartada | Motivo |
|---|---|---|
| Azure Container Instance | Azure Kubernetes Service (AKS) | AKS es overkill para una sola API; ACI = pago por segundo, cero cluster que mantener |
| ACR Basic | ACR Standard / Premium | Basic basta: 1 imagen, sin geo-replicación, ahorro x4 |
| Service Principal | Personal Access Token | Buena práctica oficial; PAT está deprecated para CI/CD |
| Imagen tag = `<commit SHA>` | `latest` | Trazabilidad y rollback al commit anterior con un solo `terraform apply` |
| Environment `production` con required reviewers | Push directo automático | La clase 24/04 lo exige: "el último clic siempre lo da un humano" |
| Variable `APP_ENV` en runtime | Imágenes distintas por entorno | Una sola imagen + env vars = principio 12-Factor; menos artefactos que mantener |

### 4.6 Trazabilidad de la arquitectura con el repositorio

| Componente del diagrama | Archivo del repo |
|---|---|
| FastAPI app | `app/main.py`, `app/models.py` |
| Imagen Docker | `Dockerfile`, `.dockerignore` |
| Resource Group, ACR, ACI | `terraform/main.tf` |
| Variables de IaC | `terraform/variables.tf` |
| Outputs (URL pública) | `terraform/outputs.tf` |
| CI (lint + tests + build) | `.github/workflows/ci.yml` |
| CD (push ACR + terraform apply) | `.github/workflows/cd.yml` |
| Tests unitarios | `tests/test_main.py` |

---
## 5. Wave Planning

El Wave Planning ordena la migración del portafolio en **3 olas (waves)** que minimizan el riesgo: empezar por aplicaciones de bajo riesgo donde el equipo aprende, y dejar las críticas para el final cuando ya se domina la herramienta.

### 5.1 Criterios de waveado
- **Wave 1 — Quick wins:** apps de bajo riesgo o con beneficio inmediato (Replatform, Repurchase, Retire). El equipo aprende contenedores e IaC.
- **Wave 2 — Datos:** apps que mueven datos (ETL). Requieren validar pipelines de datos y conectividad a BD.
- **Wave 3 — Crítico:** apps muy críticas o con datos sensibles. Sólo se mueven cuando el equipo ya tiene experiencia probada.

In [5]:
waves = [
    {
        "nombre": "Wave 1 - Quick wins",
        "duracion_semanas": 2,
        "objetivo": "Demostrar valor cloud rapido y eliminar deuda tecnica",
        "aplicaciones": [
            ("Sales Prediction API",     "Replatform", "Contenerizar + ACR + ACI + CI/CD"),
            ("Dashboard KPIs Retail",     "Repurchase", "Migrar reports a Power BI Service"),
            ("Reportes legacy clientes",  "Retire",     "Apagar; redirigir 4 usuarios a Power BI"),
        ],
        "riesgos": ["Curva de aprendizaje Terraform", "Configuracion correcta de Service Principal"],
        "salida": "1 API en produccion + 1 dashboard SaaS + 1 servidor apagado"
    },
    {
        "nombre": "Wave 2 - Datos",
        "duracion_semanas": 3,
        "objetivo": "Migrar el flujo de datos al cloud sin reescribirlo",
        "aplicaciones": [
            ("ETL nocturno de ventas", "Rehost", "Lift & shift a Azure VM + Postgres flexible"),
        ],
        "riesgos": ["Conectividad VPN entre on-prem y Azure", "Cifrado de datos sensibles en transito"],
        "salida": "Pipeline nocturno ejecutandose en Azure VM con SLA conservado"
    },
    {
        "nombre": "Wave 3 - Critico",
        "duracion_semanas": 4,
        "objetivo": "Migrar la app mas critica una vez el equipo tiene experiencia",
        "aplicaciones": [
            ("Modelo scoring de clientes", "Retain -> Refactor", "Reescribir en Python sobre Azure ML Foundry"),
        ],
        "riesgos": ["Equivalencia funcional con el modelo R legacy", "Cumplimiento normativo de datos sensibles"],
        "salida": "Modelo scoring servido como endpoint en Azure ML"
    },
]

print("WAVE PLANNING - ABC Analytics")
print("=" * 78)
for w in waves:
    print(f"\n{w['nombre']}  ({w['duracion_semanas']} semanas)")
    print("-" * 78)
    print(f"Objetivo: {w['objetivo']}")
    print("\nAplicaciones:")
    for nombre, estrategia, accion in w["aplicaciones"]:
        print(f"  - [{estrategia:18}] {nombre}")
        print(f"      Accion: {accion}")
    print("\nRiesgos clave:")
    for r in w["riesgos"]:
        print(f"  - {r}")
    print(f"\nSalida esperada: {w['salida']}")

print("\n" + "=" * 78)
total = sum(w["duracion_semanas"] for w in waves)
print(f"DURACION TOTAL ESTIMADA: {total} semanas (~{total//4} meses)")

WAVE PLANNING - ABC Analytics

Wave 1 - Quick wins  (2 semanas)
------------------------------------------------------------------------------
Objetivo: Demostrar valor cloud rapido y eliminar deuda tecnica

Aplicaciones:
  - [Replatform        ] Sales Prediction API
      Accion: Contenerizar + ACR + ACI + CI/CD
  - [Repurchase        ] Dashboard KPIs Retail
      Accion: Migrar reports a Power BI Service
  - [Retire            ] Reportes legacy clientes
      Accion: Apagar; redirigir 4 usuarios a Power BI

Riesgos clave:
  - Curva de aprendizaje Terraform
  - Configuracion correcta de Service Principal

Salida esperada: 1 API en produccion + 1 dashboard SaaS + 1 servidor apagado

Wave 2 - Datos  (3 semanas)
------------------------------------------------------------------------------
Objetivo: Migrar el flujo de datos al cloud sin reescribirlo

Aplicaciones:
  - [Rehost            ] ETL nocturno de ventas
      Accion: Lift & shift a Azure VM + Postgres flexible

Riesgos clave:
  -

### 5.2 Cronograma visual (texto)

```
Semana:    1   2   3   4   5   6   7   8   9
           │   │   │   │   │   │   │   │   │
Wave 1:    ████████                            (Sales API + Dashboard + Retire)
Wave 2:            ████████████                (ETL Rehost)
Wave 3:                        ████████████████ (Modelo scoring Refactor)
```

### 5.3 Dependencias entre waves

| Dependencia | Por qué |
|---|---|
| Wave 2 depende de Wave 1 | El equipo necesita primero practicar Terraform y CI/CD con la API antes de migrar datos |
| Wave 3 depende de Wave 2 | El modelo scoring consume datos del ETL; el ETL debe estar en Azure antes |
| Sales API es **frontera Wave 1**: la primera app que entra en producción y valida el pipeline CI/CD para todas las siguientes |

### 5.4 Criterios de éxito por wave (Definition of Done)

| Wave | DoD |
|---|---|
| **Wave 1** | `terraform apply` reproduce el entorno en menos de 5 min · CI verde en `main` · `curl /health` devuelve 200 |
| **Wave 2** | El ETL escribe en Postgres Azure el mismo número de filas que on-premise durante 7 días seguidos |
| **Wave 3** | El nuevo modelo en Azure ML produce un AUC ≥ 95 % del valor del modelo R legacy |

### 5.5 Plan de rollback por wave

- **Wave 1:** `terraform apply -var=api_image=<sha-anterior>` → vuelve a la imagen previa en menos de 2 minutos.
- **Wave 2:** mantener el ETL on-premise corriendo en paralelo durante 2 semanas (estrategia *blue-green*).
- **Wave 3:** mantener el modelo R durante 1 mes en sombra (*shadow mode*) comparando predicciones.

### 5.6 Métricas DORA esperadas tras Wave 1

| Métrica | Antes (on-prem) | Después de Wave 1 |
|---|---|---|
| Deployment Frequency | 1 / mes | 1 / sprint (cada 2 semanas) |
| Lead Time for Changes | 1 semana | < 1 día |
| Change Failure Rate | desconocido | < 10 % (medible vía workflow runs) |
| Time to Restore Service | 4 h | < 30 min (rollback con Terraform) |

---
## 6. Conclusión y aplicación al proyecto

| Aplicación | Estrategia | Wave | Acción en este repositorio |
|---|---|---|---|
| **Sales Prediction API** | **Replatform** | **Wave 1** | ✅ **Se implementa y se despliega** |
| Dashboard KPIs Retail | Repurchase | Wave 1 | Documentado, no implementado |
| Reportes legacy clientes | Retire | Wave 1 | Documentado, no implementado |
| ETL nocturno de ventas | Rehost | Wave 2 | Documentado, no implementado |
| Modelo scoring de clientes | Retain → Refactor | Wave 3 | Documentado, no implementado |

### ¿Por qué la Sales Prediction API es la app que migramos?

1. **Es la única clasificada como Replatform**, la estrategia con mejor relación esfuerzo / beneficio cloud para un equipo con experiencia media.
2. **No tiene datos sensibles**, lo que reduce el riesgo regulatorio en una primera migración.
3. **Tiene tracción real** (12 000 usuarios/mes), por lo que el beneficio operacional del cloud (auto-escalado, despliegues rápidos, IaC) se nota desde el primer día.
4. **Sirve un modelo de ML**, así que demuestra cómo aplicar **MLOps** sobre el ciclo CI/CD: build de imagen → push a ACR → terraform apply → despliegue en Azure Container Instance con aprobación humana.
5. **Encaja en Wave 1**, donde el equipo gana experiencia antes de tocar las apps más críticas.